In [1]:
%load_ext autoreload
%autoreload 2

# Get Climb Transitions

In [2]:
import networkx as nx
G = nx.read_gml("../data/graph/LEMD_EGLL_2023_04_01.gml")
idx_to_node = {i: node for i, node in enumerate(G.nodes())}
node_to_idx = {node: i for i, node in enumerate(G.nodes())}
print('Graph loaded with ', G.number_of_nodes(), ' nodes and ', G.number_of_edges(), ' edges')

Graph loaded with  566  nodes and  5028  edges


In [3]:
from test_toc_forward_dp import test_forward_dp
V_final, eta_final, alt_final, phase_final, transitions_list = test_forward_dp()

Test: Forward Dynamic Programming (forward_dp_vec2)
Using device: cpu
eps bin for ToC: 36


Topological Generations: 100%|██████████| 145/145 [00:13<00:00, 10.90it/s]



--- Results ---
V function shape: torch.Size([566, 31, 91])
Total number of unique nodes_from in transitions_list: 47
Total number of unique nodes_to in transitions_list: 145
Total number of unique transitions in transitions_list: 3259


## Inspection and Export the Transitions to a File

In [ ]:
# Plot the nodes involved in the transition_list (which are climb nodes)
from equinox.helpers.plotters import plot_waypoints 

# Get the nodes involved in the transitions by joining the source and target nodes in the transition, and make them unique 
nodes_to_highlight = list(set([node_1 for node_1, _, node_2, _ in transitions_list]))
# nodes_to_highlight.extend([node_2 for _, _, node_2, _ in transitions_list])
nodes_to_highlight = list(set(nodes_to_highlight))
nodes_to_highlight = [idx_to_node[node] for node in nodes_to_highlight]

plot_waypoints(G, highlighted_nodes=nodes_to_highlight)


In [4]:
from equinox.dp.pretoc.forward_soft_bellman import save_transitions
save_transitions(transitions_list, "../data/graph", "LEMD_EGLL_2023_04_01_climb_transitions")

Saved transitions to ../data/graph/LEMD_EGLL_2023_04_01_climb_transitions.pkl


## Transitions Inspection (skippable)

In [6]:
transitions_list

[(185, 0, 0.0, 546, 11, 14378.0),
 (185, 0, 0.0, 472, 8, 11638.0),
 (185, 0, 0.0, 491, 8, 10883.0),
 (185, 0, 0.0, 331, 11, 14575.0),
 (185, 0, 0.0, 351, 17, 20473.0),
 (185, 0, 0.0, 12, 20, 23041.0),
 (185, 0, 0.0, 255, 19, 22708.0),
 (185, 0, 0.0, 349, 23, 26760.0),
 (185, 0, 0.0, 519, 27, 29448.0),
 (185, 0, 0.0, 258, 15, 18398.0),
 (185, 0, 0.0, 513, 16, 19451.0),
 (185, 0, 0.0, 339, 20, 23397.0),
 (185, 0, 0.0, 16, 17, 19963.0),
 (185, 0, 0.0, 268, 23, 25984.0),
 (185, 0, 0.0, 447, 3, 3750.0),
 (185, 0, 0.0, 387, 20, 23040.0),
 (185, 0, 0.0, 5, 28, 30149.0),
 (185, 0, 0.0, 402, 18, 21821.0),
 (185, 0, 0.0, 469, 36, 35000.0),
 (185, 0, 0.0, 358, 26, 29086.0),
 (185, 0, 0.0, 315, 24, 27697.0),
 (185, 0, 0.0, 556, 23, 26579.0),
 (185, 0, 0.0, 102, 24, 27734.0),
 (185, 0, 0.0, 429, 29, 30737.0),
 (185, 0, 0.0, 81, 23, 25942.0),
 (185, 0, 0.0, 53, 36, 35000.0),
 (185, 0, 0.0, 350, 23, 26207.0),
 (185, 0, 0.0, 432, 19, 22369.0),
 (185, 0, 0.0, 288, 18, 20975.0),
 (185, 0, 0.0, 459, 18, 

# Test New Backward Dynamic Programming

In [ ]:
# Load the transitions from the file
import pickle 
transitions_list = pickle.load(open("../data/graph/LEMD_EGLL_2023_04_01_climb_transitions.pkl", "rb"))
print(f"Transitions list loaded with {len(transitions_list)} transitions")

In [ ]:
import networkx as nx
from equinox.cost.cost_model_1 import cost_model_1
from equinox.wind.wind_date import WindDate
from equinox.wind.wind_free import WindFree
from equinox.vnav.vnav_performance import Performance
from equinox.vnav.vnav_profiles_rev1 import NARROW_BODY_JET_CLIMB_PROFILE, NARROW_BODY_JET_DESCENT_PROFILE, NARROW_BODY_JET_CLIMB_VS_PROFILE, NARROW_BODY_JET_DESCENT_VS_PROFILE
import numpy as np
import torch

# Load the route graph
G = nx.read_gml("../data/graph/LEMD_EGLL_2023_04_01.gml")
node_to_idx = {node: i for i, node in enumerate(G.nodes())}
idx_to_node = {i: node for i, node in enumerate(G.nodes())}

node_list_for_matrix = list(G.nodes()) # Consistent order for matrix indexing
# node_to_idx_for_matrix = {nid: i for i, nid in enumerate(node_list_for_matrix)} # Not directly used in this test script main flow
num_actual_nodes = len(node_list_for_matrix)

# Load the distance matrix
dist_matrix = np.load("../data/graph/LEMD_EGLL_2023_04_01_distances.npy")

# Load the airspace charges matrix
ac_matrix = np.load("../data/graph/LEMD_EGLL_2023_04_01_charges.npy")

# Load the wind model
# Consistent with test_forward_dp, using a 2024 date for wind data,
# while flight times (landing time here) are for 2023.
wind_model = WindDate(date_str="2024-04-01", data_dir="../data/era5")
wind_model = WindFree()

# Load the performance model for a typical narrow body jet
performance_model = Performance(
    climb_speed_profile=NARROW_BODY_JET_CLIMB_PROFILE,
    descent_speed_profile=NARROW_BODY_JET_DESCENT_PROFILE,
    climb_vertical_speed_profile=NARROW_BODY_JET_CLIMB_VS_PROFILE,
    descent_vertical_speed_profile=NARROW_BODY_JET_DESCENT_VS_PROFILE,
    cruise_altitude_ft=35000.0,
    cruise_speed_kts=450.0,
)

# Load the cost model
cost_model_instance = cost_model_1

# Estimated landing time
estimated_landing_time_str = "2023-04-01 12:00:00"
# Estimated takeoff time
estimated_takeoff_time_str = "2023-04-01 10:15:00"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
print(f'ID of LEMD: {node_to_idx["LEMD"]}')
print(f'ID of EGLL: {node_to_idx["EGLL"]}')
print(f'ID of DGO: {node_to_idx["DGO"]}')
print(f'ID of NENEM: {node_to_idx["NENEM"]}')
print(f'ID of VASUX: {node_to_idx["VASUX"]}')
print(f'ID of LERM: {node_to_idx["LERM"]}')

In [ ]:
from equinox.dp.pretoc.backward_dp_vec3 import run_backward_dp
import importlib
import equinox.dp.pretoc.backward_dp_vec3
importlib.reload(equinox.dp.pretoc.backward_dp_vec3)
from equinox.dp.pretoc.backward_dp_vec3 import run_backward_dp


V, active_eta, active_alt, active_phase_return = run_backward_dp(
    graph = G,
    goal_node_id="EGLL",
    estimated_landing_time_str=estimated_landing_time_str,
    origin_elevation_ft=0.0,
    destination_elevation_ft=0.0,
    cost_model=cost_model_instance,
    wind_model=wind_model,
    performance_model=performance_model,
    dist_matrix_np=dist_matrix,
    ac_matrix_np=ac_matrix,
    transitions_list=transitions_list,
    eta_takeoff_str=estimated_takeoff_time_str,
    max_eps_bin=36,
    final_alt_ft=0.0,
    delta_t_seconds_wall_clock=300,
    delta_t_seconds_climb=30,
    max_flight_duration_hours=5,
    climb_phase_switch_allowance_wall_clock_bins=10,
    device=device,
    temperature=5e-3
)

In [ ]:
torch.sum(active_eta[node_to_idx['EGLL'], :, :, 0][~torch.isnan(active_eta[node_to_idx['EGLL'], :, :, 0])])


In [ ]:
active_eta[node_to_idx['LEMD'], :, :, 0][~torch.isnan(active_eta[node_to_idx['EGLL'], :, :, 0])]